# Collecte des données météorologiques en temps réels

Ce notebook permet de gérer les clés et l'appel API vers les stations météorologiques toutes les six minutes.

## Configuration pour l'utilisation de l'API

In [1]:
import requests
import json
import logging
import time
from datetime import datetime, timezone, timedelta
import notebookutils
from azure.eventhub import EventHubProducerClient, EventData

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 5, Finished, Available, Finished, False)

In [2]:
# Configuration pour récupérer la clé depuis azure vault keys
vault_url = "https://key-meteo.vault.azure.net/"
secret_name = "key-meteo-user"

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 6, Finished, Available, Finished, False)

In [3]:
# Récupération du secret
application_id = notebookutils.credentials.getSecret(
    vault_url,
    secret_name
)
print("Récupération de la clé secrète terminée")

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 7, Finished, Available, Finished, False)

Récupération de la clé secrète terminée


In [4]:
logging.basicConfig(level=logging.INFO)

# ==================================
# CONFIGURATION METEO FRANCE
# ==================================

API_URL = (
    'https://public-api.meteofrance.fr/public/DPPaquetObs/v2/paquet/stations/infrahoraire-6m'
)

# À renseigner selon la documentation
# d'authentification de ton abonnement
TOKEN_URL = "https://portail-api.meteofrance.fr/token"

CLIENT_SECRET = application_id 

# ==================================
# CONFIGURATION FABRIC EVENTSTREAM
# ==================================

EventHubName = "esehparzauf1zvlbk9wblge_eh"
EventHubEndpoint = "Endpoint=sb://esehparzauf1zvlbk9wblge.servicebus.windows.net/;SharedAccessKeyName=key_be1feae7-d7e4-49a0-91fb-3ef57043d053;SharedAccessKey=YZtSaJPT7CVL0lnGk4yMG4eUctmqMBQfk+AEhHZfbvM=;EntityPath=esehparzauf1zvlbk9wblge_eh"

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 8, Finished, Available, Finished, False)

## Génération automatique du token d'utilisation

Pour pouvoir gérer la génération de token automatique. 

Nous allons utiliser la clé API OAuth qui permets de générer le token pour récupérer les données.

In [5]:
headers = {
    "Authorization" : "Basic " + CLIENT_SECRET
}

def generate_token():
    """
    Cette fonction permets de générer 
    le token d'utilisation pour récupérer les informations météorologiques.
    """
    response = requests.post(
        TOKEN_URL,
        data={
            "grant_type": "client_credentials"
        },
        headers=headers,
        timeout=30
    )
    #print(response.content)
    response.raise_for_status()
    
    token_data = response.json()

    access_token = token_data["access_token"]

    expires_in = token_data.get(
        "expires_in", 86400
    )

    logging.info(
        "Nouveau token Météo-France obtenu"
    )

    return {
        "access_token": access_token,
        "expires_at": time.time() + expires_in
    }

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 9, Finished, Available, Finished, False)

In [6]:
# Faisons un test de requête pour vérifier que la requête marche
#exemple_token = generate_token()
#print(exemple_token)

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 10, Finished, Available, Finished, False)

In [7]:
#headers = {
#    "Authorization" : f"Bearer {exemple_token['access_token']}"
#}

#params = {
#    "date" : "2026-11-01T22:00:00Z",
#    "format" : "json"
#}

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 11, Finished, Available, Finished, False)

In [8]:
#response = requests.get(API_URL, headers=headers, params=params)

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 12, Finished, Available, Finished, False)

In [9]:
#response.content[0:100]

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 13, Finished, Available, Finished, False)

### Récupérer la date d'observation 

In [10]:
def get_observation_date():
    """
    Cette fonction permets de récupérer la date d'observation 
    à utiliser pour le filtre de date sur l'API d'observation des stations météorologiques
    """
    now = datetime.now(timezone.utc)

    minute = (now.minute // 6) * 6

    current_slot = now.replace(
        minute=minute,
        second=0,
        microsecond=0
    )

    observation_date = (
        current_slot - timedelta(minutes=6)
    )

    return observation_date.strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 14, Finished, Available, Finished, False)

In [11]:
#get_observation_date()

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 15, Finished, Available, Finished, False)

### Faire l'appel API pour avoir les données météorologiques

In [12]:
def get_meteo_data(token, observation_date):
    """
    Fait l'appel API en prenant en argument :
    token : le token d'utilisation pour l'appel à l'API
    observation_date : la date d'observation à utiliser dans les paramètres de l'appel
    """
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }

    params = {
        "date": observation_date,
        "format": "json"
    }

    response = requests.get(
        API_URL,
        headers=headers,
        params=params,
        timeout=120
    )

    if response.status_code == 401:
        raise PermissionError(
            "Token Météo-France expiré ou invalide"
        )

    response.raise_for_status()

    return response.json()

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 16, Finished, Available, Finished, False)

### Créer un producer EventHub

In [13]:
def create_eventhub_producer():
    """
    Permets de créer le producer d'EventHub pour la connection
    avec l'EventStream.
    """
    producer = (
        EventHubProducerClient
        .from_connection_string(
            conn_str=EventHubEndpoint,
            eventhub_name=EventHubName
            )
        )

    return producer

def publish_observations(data, observation_date):
    """
    Permets d'envoyer les observations récoltées de l'API 
    et de les envoyer avec la date filtrée :
    data : liste de json des observations des stations
    observation_date : la date d'observation utilisée pour le filtrage
    """
    producer = create_eventhub_producer()

    # Le paquet est une liste de JSON
    features = data

    if not features:
        logging.warning(
            "Aucune observation dans le paquet"
        )
        producer.close()
        return

    try:

        batch = producer.create_batch()

        for feature in features:

            event = {
                "source": "MeteoFrance",
                "observation_date": observation_date,
                "ingestion_date": datetime.now(
                    timezone.utc
                ).isoformat(),
                "data": feature
            }

            # Conversion de l'observation en JSON
            event_json = json.dumps(
                event,
                ensure_ascii=False
            )

            # Création de l'événement Event Hubs
            event_data = EventData(event_json)

            try:
                batch.add(event_data)

            except ValueError:

                # Envoi du batch lorsqu'il est plein
                producer.send_batch(batch)

                # Création d'un nouveau batch
                batch = producer.create_batch()

                batch.add(event_data)

        # Envoi du dernier batch
        if len(batch) > 0:
            producer.send_batch(batch)
        msg = f"{len(features)} observations publiées"
        print(msg)
        logging.info(
            msg
        )

    finally:
        producer.close()

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 17, Finished, Available, Finished, False)

### Exécution de toutes les parties en une seule fonction

In [14]:
def main():
    """
    La fonction principale pour faire l'exécution de toutes les étapes :
    (1) - récupérer la date d'observation
    (2) - générer le token d'utilisation
    (3) - appel API avec le token généré et la date d'observation
    (4) - envoie du JSON résultant au producer d'EventHub (liée à l'EventStream 'Eventstream_meteo_france')
    """
    observation_date = get_observation_date()

    logging.info(
        f"Collecte des observations : {observation_date}"
    )

    token_info = generate_token()

    try:

        data = get_meteo_data(
            token_info["access_token"],
            observation_date
        )

    except PermissionError:

        logging.warning(
            "Token refusé : renouvellement"
        )

        token_info = generate_token()

        data = get_meteo_data(
            token_info["access_token"],
            observation_date
        )
    #print(data)
    publish_observations(
        data,
        observation_date
    )


StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 18, Finished, Available, Finished, False)

In [15]:
main()

StatementMeta(, 667c3d2c-431f-437f-94d8-990aa4c8bcde, 19, Finished, Available, Finished, False)

1762 observations publiées
